In [44]:
import os
from itertools import count

import cv2

os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

from config import HOW2SIGN_RAW_DATA, WLASL_RAW_DATA
from src.utils import HandDetection, PoseDetection


def resize_with_padding(frame, target_size):
    """Resize giữ nguyên tỷ lệ khung hình, pad thêm viền đen để thành hình vuông.
    Không cắt mất bất kỳ nội dung nào của frame gốc."""
    h, w = frame.shape[:2]
    scale = target_size / max(h, w)
    new_w, new_h = int(round(w * scale)), int(round(h * scale))
    resized = cv2.resize(frame, (new_w, new_h), interpolation=cv2.INTER_LINEAR)

    pad_w = target_size - new_w
    pad_h = target_size - new_h
    top, bottom = pad_h // 2, pad_h - pad_h // 2
    left, right = pad_w // 2, pad_w - pad_w // 2

    padded = cv2.copyMakeBorder(
        resized, top, bottom, left, right,
        borderType=cv2.BORDER_CONSTANT,
        value=(0, 0, 0),
    )
    return padded


input_video = os.path.join(WLASL_RAW_DATA, "00335.mp4")
output_video = os.path.join(r"D:\SignDetection", "00335.mp4")

cap = cv2.VideoCapture(input_video)

if not cap.isOpened():
    raise RuntimeError("Cannot open video.")

hand_detection = HandDetection()
pose_detection = PoseDetection()

fps = cap.get(cv2.CAP_PROP_FPS)
if fps <= 0:
    fps = 25

width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
print(f"width: {width}, height: {height}")

fourcc = cv2.VideoWriter_fourcc(*"mp4v")

# 256: khớp Pose Landmarker (256x256), đủ lớn để Hand Landmarker (192/224) downscale không mất chi tiết
target_size = 256

video_writer = cv2.VideoWriter(
    output_video,
    fourcc,
    fps,
    (target_size, target_size),
)

frame_index = 0
count = 0
while True:
    success, frame = cap.read()

    if not success:
        print("End of video.")
        break

    # Resize + pad thay vì crop -> giữ nguyên toàn bộ nội dung, không mất tay/cử chỉ
    frame = resize_with_padding(frame, target_size)

    rgb_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)

    timestamp_ms = int(frame_index * 1000 / fps)

    detection_hand_results = hand_detection.detect_video(rgb_frame, timestamp_ms)
    detection_pose_results = pose_detection.detect_video(rgb_frame, timestamp_ms)
    print(f"frame_index: {frame_index} {detection_hand_results}")
    output = hand_detection.draw_landmarks_on_image(rgb_frame.copy(), detection_hand_results)
    output = pose_detection.draw_landmarks_on_image(output, detection_pose_results)

    output = cv2.cvtColor(output, cv2.COLOR_RGB2BGR)

    video_writer.write(output)
    cv2.imshow("frame", output)

    frame_index += 1
    count += 1

    if cv2.waitKey(1) & 0xFF == ord("q"):
        break

cap.release()
video_writer.release()
cv2.destroyAllWindows()
hand_detection.close()
print(f"Saved video to: {output_video}")

width: 320, height: 240
frame_index: 0 HandLandmarkerResult(handedness=[], hand_landmarks=[], hand_world_landmarks=[])
frame_index: 1 HandLandmarkerResult(handedness=[], hand_landmarks=[], hand_world_landmarks=[])
frame_index: 2 HandLandmarkerResult(handedness=[], hand_landmarks=[], hand_world_landmarks=[])
frame_index: 3 HandLandmarkerResult(handedness=[], hand_landmarks=[], hand_world_landmarks=[])
frame_index: 4 HandLandmarkerResult(handedness=[], hand_landmarks=[], hand_world_landmarks=[])
frame_index: 5 HandLandmarkerResult(handedness=[], hand_landmarks=[], hand_world_landmarks=[])
frame_index: 6 HandLandmarkerResult(handedness=[], hand_landmarks=[], hand_world_landmarks=[])
frame_index: 7 HandLandmarkerResult(handedness=[], hand_landmarks=[], hand_world_landmarks=[])
frame_index: 8 HandLandmarkerResult(handedness=[], hand_landmarks=[], hand_world_landmarks=[])
frame_index: 9 HandLandmarkerResult(handedness=[], hand_landmarks=[], hand_world_landmarks=[])
frame_index: 10 HandLandma